# SQ1 preprocessing: donor-stratified subsample for SeaAD paired multiome

The team's existing Phase 2 (`feature_selection.py`) subsamples **50 cells per cell type without stratifying by donor**, producing `_sub42` files with ~50 cells per cell type spread across all 28 multiome donors (≈2 cells per donor × cell type). That's correct for the precision/recall benchmark but **too few cells per donor for SQ1's per-donor pseudobulk regression**.

This notebook produces an SQ1-specific subsample:

* Filter to target cell types only (`Microglia-PVM`, `L2/3 IT`).
* Filter to donors with a defined AD/control condition and ≥50 cells in the cell type.
* Subsample **50 cells per (donor × cell type)** — donor-stratified, the right unit for SQ1.
* Run Phase 2 (HVG/HVP + KNN) and Phase 3 (gene-peak triplets) on that subsample.
* Write outputs with a `sq1` tag so they don't collide with the team's `_sub42` files.

After this notebook, set `SUB_TAG = "sq1"` in `run_infer_wscreni_seaad.ipynb` and it runs on the SQ1 subsample.

**Wall time:** dominated by the 94 GB integrated h5mu read (~5–30 min first time, faster on warm cache) and Phase 3 motif scanning (~30 min – 2 h depending on subsample size). Submit via SLURM rather than running interactively.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import h5py
import anndata as ad

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

from screni.data.feature_selection import prepare_subsample
from screni.data.gene_peak_relations import load_transfac_motifs, run_phase3
from screni.data.loading_seaad import (
    add_condition_column,
    add_copathology_columns,
    select_eligible_donors,
    subsample_cells_per_donor,
)
from screni.data.utils import load_gene_annotations

In [ ]:
# ---- Configuration ----
DATASET            = "seaad_paired"
SUB_TAG            = "sq1"
TARGET_CELL_TYPES  = ["Microglia-PVM", "L2/3 IT"]
N_CELLS_PER_DONOR  = 50
MIN_CELLS_PER_DONOR = 50              # donor must have ≥50 cells in the type to be included
N_GENES            = 2000             # widened from 500 to capture lower-variance regulatory TFs
                                       #   (SPI1/IRF8/CEBPB for microglia, REST/MEF2C for neurons)
                                       #   that the top-500 HVG cut had missed
N_PEAKS            = 10000            # HVPs for Phase 2
KNN_K              = 20
SEED               = 42

DATA_DIR     = os.path.join(REPO_ROOT, "data", "processed", "seaad")
REF_DIR      = os.path.join(REPO_ROOT, "data", "paper", "reference")
GENOME_FA    = os.path.join(REPO_ROOT, "data", "reference", "hg38.fa")

RNA_FULL     = os.path.join(DATA_DIR, f"{DATASET}_rna.h5ad")
ATAC_FULL    = os.path.join(DATA_DIR, f"{DATASET}_atac.h5ad")
INTEGRATED   = os.path.join(DATA_DIR, f"{DATASET}_integrated.h5mu")

GTF_PATH     = os.path.join(REF_DIR, "gencode.v38.annotation.gtf")
MOTIF_TXT    = os.path.join(REF_DIR, "Tranfac201803_Hs_MotifTFsFinal")
MOTIF_RDS    = os.path.join(REF_DIR, "all_motif_pwm.rds")

for label, path in [
    ("RNA full", RNA_FULL), ("ATAC full", ATAC_FULL), ("integrated", INTEGRATED),
    ("GTF", GTF_PATH), ("motif TSV", MOTIF_TXT), ("motif RDS", MOTIF_RDS),
    ("genome FA", GENOME_FA),
]:
    print(f"  {'✓' if os.path.exists(path) else '✗'} {label}: {path}")

In [ ]:
# Step 1: load the FULL paired RNA + ATAC (these are 12.5 GB and 16.8 GB).
# Tag condition + co-pathology + filter to target cell types and cells with
# a defined ad/control condition. Aligning RNA <-> ATAC by barcode comes after.
print("loading RNA ...")
rna = ad.read_h5ad(RNA_FULL)
print(f"  RNA: {rna.shape}")

print("loading ATAC ...")
atac = ad.read_h5ad(ATAC_FULL)
print(f"  ATAC: {atac.shape}")

# Standardise cell_type column from Subclass
if "cell_type" not in rna.obs.columns:
    rna.obs["cell_type"] = rna.obs["Subclass"].astype(str)
if "cell_type" not in atac.obs.columns:
    atac.obs["cell_type"] = atac.obs["Subclass"].astype(str)

add_condition_column(rna)
add_copathology_columns(rna)
add_condition_column(atac)
add_copathology_columns(atac)

ct_mask  = rna.obs["cell_type"].isin(TARGET_CELL_TYPES) & rna.obs["condition"].notna()
rna_f    = rna[ct_mask].copy()
atac_f   = atac[atac.obs["cell_type"].isin(TARGET_CELL_TYPES) & atac.obs["condition"].notna()].copy()
print(f"after target-cell-type + condition filter: RNA={rna_f.shape}, ATAC={atac_f.shape}")
print(f"  cell_type counts (RNA): {rna_f.obs['cell_type'].value_counts().to_dict()}")

In [ ]:
# Step 2: identify eligible donors per cell type (≥50 cells of that type with a
# defined condition). This is the SQ1 cohort.
elig_tables = {}
for ct in TARGET_CELL_TYPES:
    elig_tables[ct] = select_eligible_donors(
        rna_f,
        cell_type=ct,
        min_cells_per_donor=MIN_CELLS_PER_DONOR,
        cell_type_col="cell_type",
        donor_col="Donor ID",
        require_condition=True,
    )
    print(f"\n{ct}: {len(elig_tables[ct])} eligible donors")
    print(elig_tables[ct].to_string(index=False))

# Union of eligible donors across both target cell types
eligible_donors = sorted(set().union(*[set(t["donor_id"]) for t in elig_tables.values()]))
print(f"\nunion of eligible donors: {len(eligible_donors)}")

In [ ]:
# Step 3: filter RNA + ATAC to eligible donors only, then donor-stratified
# subsample (50 cells per donor × cell_type). Pair RNA <-> ATAC by barcode
# at the end.
rna_e  = rna_f[rna_f.obs["Donor ID"].isin(eligible_donors)].copy()
atac_e = atac_f[atac_f.obs["Donor ID"].isin(eligible_donors)].copy()

rna_sub = subsample_cells_per_donor(
    rna_e, n_per_donor=N_CELLS_PER_DONOR,
    donor_col="Donor ID", cell_type=None,    # already filtered by cell type above
    cell_type_col="cell_type", seed=SEED,
)

# Pair ATAC to the subsampled RNA cells
shared = sorted(set(rna_sub.obs_names) & set(atac_e.obs_names))
if len(shared) < rna_sub.n_obs:
    print(f"WARN: lost {rna_sub.n_obs - len(shared)} cells in RNA<->ATAC pairing")
rna_sub  = rna_sub[shared].copy()
atac_sub = atac_e[shared].copy()
print(f"\nfinal subsample: RNA={rna_sub.shape}, ATAC={atac_sub.shape}")
print(f"  (donor, cell_type) counts:")
print(rna_sub.obs.groupby(['Donor ID', 'cell_type'], observed=True).size())

In [ ]:
# Step 4: pull the WNN/PCA embedding from the integrated h5mu for the
# subsampled cells only. Using h5py + indexing avoids loading the 94 GB file
# fully into memory.
EMB_KEY  = "X_pca"     # team's integration stores the joint multimodal embedding under this key
MOD_NAME = "rna"

with h5py.File(INTEGRATED, "r") as f:
    obs_names_full = [n.decode() if isinstance(n, bytes) else n
                       for n in f[f"mod/{MOD_NAME}/obs/_index"][:]]
    name_to_row = {n: i for i, n in enumerate(obs_names_full)}
    rows = [name_to_row[c] for c in rna_sub.obs_names if c in name_to_row]
    if len(rows) != rna_sub.n_obs:
        missing = set(rna_sub.obs_names) - set(obs_names_full)
        raise RuntimeError(f"{len(missing)} subsampled cells not in integrated h5mu "
                           f"(e.g. {sorted(missing)[:3]})")
    emb_full = f[f"mod/{MOD_NAME}/obsm/{EMB_KEY}"]
    embedding = np.asarray(emb_full[rows, :])
    embedding_cell_names = list(rna_sub.obs_names)
print(f"embedding pulled: {embedding.shape} (from key mod/{MOD_NAME}/obsm/{EMB_KEY})")

In [ ]:
# Step 5: Phase 2 — HVG/HVP feature selection + KNN.
# We've already subsampled cells, so pass an effectively-no-op n_per_type.
phase2 = prepare_subsample(
    rna                  = rna_sub,
    atac                 = atac_sub,
    n_per_type           = 10**9,          # already subsampled per donor × type
    n_genes              = N_GENES,
    n_peaks              = N_PEAKS,
    seed                 = SEED,
    embedding            = embedding,
    embedding_cell_names = embedding_cell_names,
    knn_k                = KNN_K,
)
rna_p2  = phase2["rna"]
atac_p2 = phase2["atac"]
knn     = phase2["knn_indices"]
print(f"\nPhase 2 done: RNA={rna_p2.shape}, ATAC={atac_p2.shape}, KNN={knn.shape}")

# Embed KNN into the RNA h5ad's uns so the inference notebook reads it from there
rna_p2.uns["knn_indices"] = knn

In [ ]:
# Step 6: write Phase 2 outputs with the sq1 tag.
rna_out  = os.path.join(DATA_DIR, f"{DATASET}_rna_{SUB_TAG}.h5ad")
atac_out = os.path.join(DATA_DIR, f"{DATASET}_atac_{SUB_TAG}.h5ad")
rna_p2.write_h5ad(rna_out)
atac_p2.write_h5ad(atac_out)
print(f"wrote: {rna_out}")
print(f"wrote: {atac_out}")

In [ ]:
# Step 7: Phase 3 — gene-peak-TF triplets on the SQ1 subsample.
# Uses the team's run_phase3() with the raw GENCODE GTF (the existing
# load_gene_annotations parses it correctly).
print("loading gene annotations ...")
gene_ann = load_gene_annotations(GTF_PATH)
print(f"  loaded {len(gene_ann)} gene records")

print("loading TRANSFAC motifs + PWMs ...")
pwm_dict, motif_db = load_transfac_motifs(MOTIF_RDS, MOTIF_TXT, gene_name_type="symbol")
print(f"  motifs: {len(motif_db)}, PWMs: {len(pwm_dict)}")

phase3 = run_phase3(
    rna_adata        = rna_p2,
    atac_adata       = atac_p2,
    gene_annotations = gene_ann,
    genome_fasta     = GENOME_FA,
    pwm_dict         = pwm_dict,
    motif_db         = motif_db,
    gene_name_type   = "symbol",
    output_dir       = DATA_DIR,
    prefix           = f"{DATASET}_{SUB_TAG}",   # writes seaad_paired_sq1_*.csv etc.
)
print("\nPhase 3 done:")
print(f"  triplets:           {phase3['triplets'].shape}")
print(f"  gene_labels:        {phase3['gene_labels'].shape}")
print(f"  peak_overlap_matrix: {phase3['peak_matrix'].shape}")

## Summary of artifacts written

All under `data/processed/seaad/`:

| File | Contents |
|---|---|
| `seaad_paired_rna_sq1.h5ad` | Donor-stratified subsample, 500 HVGs, KNN in `.uns['knn_indices']` |
| `seaad_paired_atac_sq1.h5ad` | Same cells, 10000 HV peaks |
| `seaad_paired_sq1_triplets.csv` | TF → peak → target_gene with Spearman r |
| `seaad_paired_sq1_gene_labels.csv` | gene type (TF/target) + associated peaks/TFs |
| `seaad_paired_sq1_peak_overlap_matrix.npz` | (cells × n_unique_peaks) peak accessibility matrix |
| `seaad_paired_sq1_peak_gene_pairs.csv` | correlated peak-gene pairs |
| `seaad_paired_sq1_motif_peak_pairs.csv` | motif matches per peak |
| `seaad_paired_sq1_peak_info.csv` | peak → gene mapping |

**Next step:** open `run_infer_wscreni_seaad.ipynb`, change `SUB_TAG = "sq1"`, and run the inference. It picks up these files automatically.